In [1]:
# Core libraries for this lecture
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the CSV into a DataFrame.
# If this ever raises UnicodeDecodeError on another file, try encoding="latin1".
df = pd.read_csv("Sample-Superstore2019.csv")

# Quick confirmation that the file loaded and how big it is (rows, columns)
df.shape

(9994, 22)

In [ ]:
# Shape: (number of rows, number of columns)
print("Shape:", df.shape)

# First and last rows — sanity check that data looks like what we expect
df.head()

In [ ]:
df.tail(3)

In [ ]:
# Column names
print(df.columns.tolist())

# .info() gives dtypes + non-null counts + memory usage all at once
df.info()

In [ ]:
# TODO 1: print the column names as a list


# TODO 2: display the first 7 rows


# TODO 3: inspect df.info() output and note the column(s) with missing values (write your answer as a comment)


# TODO 4: re-read the CSV using a read_csv parameter that avoids the Unnamed: 0 column


In [ ]:
# Summary stats for numeric columns: Sales, Quantity, Discount, Profit, Postal Code, Row ID...
df.describe()

In [ ]:
# Summary stats for text/categorical columns
df.describe(include="str")

In [ ]:
# How many orders per Region / Segment / Ship Mode?
print(df["Region"].value_counts())
print()
print(df["Segment"].value_counts())
print()
print(df["Ship Mode"].value_counts())

In [ ]:
# TODO 1: describe() on Sales and Profit only


# TODO 2: value_counts() on Category


# TODO 3: value_counts(normalize=True) on Ship Mode


# TODO 4: comment on whether Discount's min/max look plausible


In [ ]:
# Convert the two date columns from text to real datetime objects
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Now that they're real dates, we can do date arithmetic:
# how many days did each order take to ship?
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

df[["Order Date", "Ship Date", "Shipping Days"]].head()

In [ ]:
# Pull the order month/year out now that Order Date is a real datetime
df["Order Month"] = df["Order Date"].dt.month
df["Order Year"] = df["Order Date"].dt.year

# Convert low-cardinality text columns to the 'category' dtype (memory + speed win)
for col in ["Segment", "Region", "Ship Mode", "Category"]:
    df[col] = df[col].astype("category")

df.dtypes

In [ ]:
# TODO 1: confirm Order Date's dtype


# TODO 2: create an "Order Weekday" column using .dt.day_name()


# TODO 3: compute the average Shipping Days


# TODO 4: convert Sub-Category to category dtype


In [ ]:
# Count missing values per column
df.isnull().sum()

In [ ]:
# Look at the actual rows with a missing Postal Code — is there a pattern?
df[df["Postal Code"].isnull()][["City", "State", "Postal Code"]]

In [ ]:
# All 11 missing postal codes belong to Burlington, Vermont.
# That's not random -- we can look up the real ZIP code (05401) and fill it in
# rather than dropping 11 otherwise-perfectly-good rows.
df["Postal Code"] = df["Postal Code"].fillna(5401)

# Confirm there are no missing values left in this column
df["Postal Code"].isnull().sum()

In [ ]:
# TODO 1: confirm no missing values remain


# TODO 2: write (but don't overwrite df with) a dropna() call for Postal Code


# TODO 3: comment — when is dropping rows the wrong call?


In [ ]:
# Exact full-row duplicates
print("Exact duplicate rows:", df.duplicated().sum())

# Duplicates on a more meaningful subset: same order line for the same product
print("Duplicate Order ID + Product ID pairs:", df.duplicated(subset=["Order ID", "Product ID"]).sum())

In [ ]:
# Inspect what those subset-duplicates actually look like
dup_mask = df.duplicated(subset=["Order ID", "Product ID"], keep=False)
df[dup_mask].sort_values(["Order ID", "Product ID"]).head(8)

In [ ]:
# TODO 1: duplicated() on Customer ID + Order Date


# TODO 2: comment explaining keep="first" vs keep="last"


# TODO 3: name (and try) the method that would actually remove duplicate rows


In [ ]:
# Check every key categorical column for stray variants (casing, whitespace, typos)
for col in ["Segment", "Region", "Ship Mode", "Category", "Country/Region"]:
    print(col, "->", df[col].unique())

In [ ]:
# Even on a clean column, this is the defensive pattern to reach for
# whenever you *do* find inconsistent casing/whitespace:
df["Segment"] = df["Segment"].astype(str).str.strip().str.title()
df["Segment"].unique()

In [ ]:
# TODO 1: unique() on Sub-Category


# TODO 2: clean the simulated dirty Series into one consistent value


# TODO 3: comment — why does inconsistent text break groupby later?


In [ ]:
# Range sanity check: Discount should never be negative or above 1 (100%)
print("Discount min/max:", df["Discount"].min(), df["Discount"].max())

# Negative profit is common in this dataset -- a real business signal, not an error
print("Orders sold at a loss:", (df["Profit"] < 0).sum())

In [ ]:
# Visual check for outliers in Sales using a boxplot
plt.figure(figsize=(6, 4))
plt.boxplot(df["Sales"], orientation="horizontal")
plt.title("Sales distribution -- boxplot")
plt.xlabel("Sales")
plt.show()

In [ ]:
# IQR method: flag numeric outliers beyond 1.5x the interquartile range
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df["Sales"] < lower_bound) | (df["Sales"] > upper_bound)]
print(f"Sales outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print("Number of Sales outliers:", len(outliers))

In [ ]:
# TODO 1: IQR outlier method applied to Profit


# TODO 2: boxplot for Quantity


# TODO 3: range sanity check on Quantity (min/max)


In [ ]:
# Boolean indexing: orders sold at a loss
loss_orders = df[df["Profit"] < 0]
print("Loss-making orders:", len(loss_orders))
loss_orders[["Order ID", "Product Name", "Sales", "Profit"]].head()

In [ ]:
# Combine multiple conditions with & (remember the parentheses around each condition!)
same_day_west = df[(df["Ship Mode"] == "Same Day") & (df["Region"] == "West")]
same_day_west[["Order ID", "Region", "Ship Mode", "Sales"]].head()

In [ ]:
# .loc for label-based selection: rows where Profit < 0, only a few relevant columns
df.loc[df["Profit"] < 0, ["Order ID", "Category", "Sub-Category", "Profit"]].head()

# .iloc for pure position-based selection: first 5 rows, first 3 columns
df.iloc[:5, :3]

In [ ]:
# TODO 1: Technology orders with Sales > 1000


# TODO 2: .loc selection -- Order ID, Customer Name, Profit where Discount > 0.5


# TODO 3: .iloc -- last 5 rows, last 4 columns


# TODO 4: combine three conditions with &


In [ ]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

In [ ]:
# Multiple aggregations at once with .agg()
category_summary = df.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

In [ ]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

In [ ]:
# TODO 1: total Profit per Sub-Category, sorted ascending


# TODO 2: pivot table -- Sales by Segment (rows) x Ship Mode (columns)


# TODO 3: average and max Shipping Days per Ship Mode


In [ ]:
# TODO 1: State with highest total Profit, and State with largest total loss


# TODO 2: relationship between Discount (binned) and average Profit


# TODO 3: your own exploration
